# Implementasi ELT Pipeline — Prediksi Ketersediaan Sepeda Citi Bike Jersey City
**Mata kuliah:** Big Data · **Tugas Minggu 11** · **Kelompok 14**
Herpratama Rafi Putranto (101032300076) · Fandy Rifky Melani Saputra (101032300181)

Notebook ini menjalankan pipeline **Extract → Load → Transform** (ELT). Data mentah dimuat dahulu ke zona *raw* di target storage,
lalu seluruh pembersihan dan transformasi dijalankan sebagai **SQL di dalam PostgreSQL** terhadap data yang sudah tersimpan.
Kode berada di repositori https://github.com/rafiptrnto/citibike-jc-pipeline (paket `elt/`).

| Komponen | Isi |
|---|---|
| Sumber data | Trip data bulanan Citi Bike Jersey City (S3, CSV dalam ZIP), GBFS Citi Bike (API JSON), cuaca Open-Meteo (API JSON) |
| Target storage | PostgreSQL pada Supabase (proyek `citibike-etl`), sama dengan Minggu 10 |
| Zona raw | schema `raw_elt` (data mentah apa adanya, tidak pernah diubah oleh Transform) |
| Zona hasil transformasi | schema `elt` (`trips_clean`, `gbfs_status_clean`, `dim_station`, `weather_hourly`, `fact_station_30min`) |
| Pembanding | schema `public` = hasil pipeline ETL Minggu 10 |
| Bulan yang didemonstrasikan | Oktober 2026 (`202610`) + satu snapshot GBFS terbaru |

Urutan: **1.** Persiapan · **2.** Extract · **3.** Load raw · **4.** Transform · **5.** Integrasi pipeline · **6.** Testing · **7.** Dataset.

## 1. Persiapan lingkungan dan koneksi ke target storage

In [ ]:
import os, sys, subprocess, getpass, datetime as dt

if os.path.basename(os.getcwd()) != "citibike-jc-pipeline":
    if not os.path.exists("citibike-jc-pipeline"):
        subprocess.run(["git", "clone", "-q", "https://github.com/rafiptrnto/citibike-jc-pipeline.git"], check=True)
    os.chdir("citibike-jc-pipeline")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psycopg2-binary"], check=True)
sys.path.insert(0, os.getcwd())
print("Kode pipeline siap di:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Connection string Supabase (Session pooler) dimasukkan saat dijalankan, tidak disimpan di notebook.
os.environ["DATABASE_URL"] = getpass.getpass("Connection string Supabase: ")

import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
from etl import config
from elt import extract as X, load as L, transform as TR, inspect_raw as INS, run_elt

conn = L.connect()
print("KONEKSI KE TARGET STORAGE BERHASIL")
display(L.query_df(conn, "select current_database() as database, current_user as pengguna, split_part(version(), ' ', 2) as versi_postgresql"))
display(L.query_df(conn, """
    select table_schema as schema, table_name as tabel
    from information_schema.tables where table_schema in ('raw_elt','elt') order by 1, 2"""))

## 2. Extract — pengambilan data dari sumber (tanpa cleaning)

Tiga sumber diekstraksi dengan `elt/extract.py`. Berbeda dengan ETL, **tidak ada konversi tipe, penyaringan, atau pembersihan**:
semua nilai dipertahankan sebagai teks persis seperti sumbernya (trip data dan cuaca), dan setiap stasiun GBFS dipertahankan sebagai objek JSON aslinya.

In [ ]:
def inspeksi(df, nama, n=5):
    """Pemeriksaan awal (bukan cleaning): jumlah baris/kolom, tipe, missing values, duplikat, contoh baris."""
    print(f"===== {nama} =====")
    print(f"Jumlah baris : {len(df):,}")
    print(f"Jumlah kolom : {df.shape[1]}")
    d = df.replace("", pd.NA)
    ringkas = pd.DataFrame({"tipe_data": df.dtypes.astype(str), "kosong": d.isna().sum(),
                            "kosong_%": (d.isna().mean() * 100).round(2), "nilai_unik": df.astype(str).nunique()})
    display(ringkas)
    dup = int(df.astype(str).duplicated().sum())
    print(f"Baris duplikat penuh : {dup:,}")
    print(f"Total sel kosong     : {int(d.isna().sum().sum()):,}")
    display(df.head(n))
    return {"baris": len(df), "kolom": df.shape[1], "sel_kosong": int(d.isna().sum().sum()), "duplikat_penuh": dup}

BULAN = "202610"
print(f"Proses Extract dimulai ({BULAN})")
trips_raw, sumber_trips = X.extract_trips_raw(BULAN)
assert trips_raw is not None, "File trip bulan ini belum tersedia di S3"
print("Sumber trip data:", sumber_trips)
r_trips = inspeksi(trips_raw, f"Trip data {BULAN} (S3, CSV dalam ZIP)")

In [ ]:
info_rec, status_rec, snap, feeds = X.extract_gbfs_raw()
print("Snapshot GBFS (epoch):", snap, "->", pd.to_datetime(int(snap), unit="s", utc=True))
r_info = inspeksi(pd.DataFrame(info_rec), "GBFS station_information (API JSON)")
r_status = inspeksi(pd.DataFrame(status_rec), "GBFS station_status (API JSON)")

In [ ]:
awal_bln = pd.Timestamp(f"{BULAN[:4]}-{BULAN[4:]}-01")
akhir_bln = awal_bln + pd.offsets.MonthBegin(1)
cuaca_raw = X.extract_weather_raw(awal_bln.date(), (akhir_bln - pd.Timedelta(days=1)).date(),
                                  today=pd.Timestamp.now(tz=config.TZ).date())
r_cuaca = inspeksi(cuaca_raw, "Cuaca Open-Meteo (API JSON)")
print("Data berhasil diekstraksi (dalam kondisi mentah)")
kondisi_awal = pd.DataFrame([
    [f"Trip data {BULAN}", "CSV dalam ZIP (S3)", r_trips["baris"], r_trips["kolom"], r_trips["sel_kosong"], r_trips["duplikat_penuh"]],
    ["GBFS station_information", "API JSON (GBFS 2.3)", r_info["baris"], r_info["kolom"], r_info["sel_kosong"], r_info["duplikat_penuh"]],
    ["GBFS station_status", "API JSON (GBFS 2.3)", r_status["baris"], r_status["kolom"], r_status["sel_kosong"], r_status["duplikat_penuh"]],
    ["Cuaca Open-Meteo", "API JSON", r_cuaca["baris"], r_cuaca["kolom"], r_cuaca["sel_kosong"], r_cuaca["duplikat_penuh"]],
], columns=["Sumber", "Format", "Baris", "Kolom", "Sel kosong", "Duplikat penuh"])
display(kondisi_awal)

## 3. Load — memuat data mentah ke zona raw (`raw_elt`)

**Struktur penyimpanan raw.** Satu tabel raw per sumber di schema `raw_elt`, dipisahkan dari hasil transformasi (schema `elt`):

| Tabel raw | Isi | Struktur |
|---|---|---|
| `raw_elt.trips` | 13 kolom file CSV | semua kolom bertipe `text` (tanpa konversi), ditambah `batch_id`, `row_no` |
| `raw_elt.gbfs_station_information` | satu baris per stasiun | objek JSON asli (`jsonb`) |
| `raw_elt.gbfs_station_status` | satu baris per stasiun + waktu snapshot | objek JSON asli (`jsonb`), `snapshot_time` |
| `raw_elt.weather` | respons cuaca per jam | nilai sebagai `text`, ditambah `sumber_api` |
| `raw_elt.load_batch` | jejak setiap pemuatan | sumber, jumlah baris diekstraksi, jumlah baris tersimpan, waktu |

Data dimuat dengan `COPY` (cepat). Tabel raw bersifat **append-only**: file trip bulanan bersifat tetap, sehingga bulan yang sudah termuat lengkap tidak dimuat ulang.

In [ ]:
display(L.query_df(conn, """
    select table_name as tabel, ordinal_position as no, column_name as kolom, data_type as tipe
    from information_schema.columns where table_schema = 'raw_elt' and table_name in
      ('trips','gbfs_station_information','gbfs_station_status','weather') order by table_name, ordinal_position"""))

In [ ]:
print("Proses Load dimulai: memuat data mentah ke raw_elt (tanpa cleaning)")
b_trips, n_trips, trips_sudah_ada = L.load_trips_raw(conn, trips_raw, BULAN, sumber_trips)
b_w, n_w = L.load_weather_raw(conn, cuaca_raw, config.ARCHIVE_URL)
(b_i, n_i), (b_s, n_s) = L.load_gbfs_raw(conn, info_rec, status_rec, snap, feeds)
conn.commit()
print(f"Trip data   : batch {b_trips}, {n_trips:,} baris" + (" (bulan ini sudah termuat sebelumnya oleh GitHub Actions; tidak dimuat ulang)" if trips_sudah_ada else ""))
print(f"Cuaca       : batch {b_w}, {n_w:,} baris")
print(f"GBFS info   : batch {b_i}, {n_i:,} baris | GBFS status: batch {b_s}, {n_s:,} baris")
print("Raw data berhasil dimuat ke storage")
display(L.query_df(conn, """select batch_id, sumber, bulan, baris_extract, baris_load, loaded_at, github_run_id
                            from raw_elt.load_batch order by batch_id"""))

### 3.1 Pemeriksaan hasil penyimpanan raw

In [ ]:
cek_raw = pd.DataFrame([
    ["Trip data", len(trips_raw), n_trips],
    ["Cuaca", len(cuaca_raw), n_w],
    ["GBFS station_information", len(info_rec), n_i],
    ["GBFS station_status", len(status_rec), n_s]], columns=["Sumber", "Baris hasil ekstraksi", "Baris tersimpan di raw"])
cek_raw["Sama"] = cek_raw["Baris hasil ekstraksi"] == cek_raw["Baris tersimpan di raw"]
display(cek_raw)
assert cek_raw["Sama"].all()

# Struktur dan atribut dapat diakses; data dibaca kembali dari target storage
raw_baca = L.query_df(conn, "select * from raw_elt.trips where batch_id = %s order by row_no limit 5", (b_trips,))
display(raw_baca)
display(L.query_df(conn, """select row_no, payload->>'station_id' as station_id, payload->>'short_name' as short_name,
                            payload->>'name' as name, payload->>'capacity' as capacity
                            from raw_elt.gbfs_station_information where batch_id = %s order by row_no limit 5""", (b_i,)))

# Raw dibaca kembali dan dibandingkan dengan hasil ekstraksi (nilai teks identik, kosong = NULL)
baca_penuh = L.query_df(conn, "select ride_id, started_at, ended_at, start_station_id, end_station_id from raw_elt.trips where batch_id = %s order by row_no", (b_trips,))
kol_cek = ["ride_id", "started_at", "ended_at", "start_station_id", "end_station_id"]
asli = trips_raw[kol_cek].astype(object).where(trips_raw[kol_cek] != "", None).reset_index(drop=True)
identik = bool((baca_penuh.fillna("<null>").values == asli.fillna("<null>").values).all())
print(f"Raw terbaca kembali: {len(baca_penuh):,} baris; isi identik dengan hasil ekstraksi: {identik}")
assert identik

## 4. Transform — pembersihan dan transformasi di dalam database

Transformasi dijalankan sebagai SQL **di dalam PostgreSQL** terhadap data raw yang sudah tersimpan (`elt/transform.py`); Python hanya mengirim perintah SQL
dan tidak menarik data untuk diolah. Aturan sama dengan pipeline ETL agar hasilnya dapat dibandingkan:

| Kode | Aturan (SQL) | Alasan |
|---|---|---|
| R1 | Saring stasiun dengan `short_name` JC/HB, kapasitas > 0, `is_installed` | Studi kasus hanya Jersey City dan Hoboken |
| R2 | Petakan ID live (UUID) ke ID historical lewat `short_name`; cadangan lewat nama unik | Kunci stasiun harus sama di kedua sumber |
| R3 | Regex + `::timestamp` (teks → datetime), UTC → America/New_York, `date_bin('30 minutes')` | Tipe data benar dan granularitas slot 30 menit |
| R4 | `DISTINCT ON (ride_id)`; `DISTINCT ON (snapshot_time, station_id)` | Hindari perhitungan ganda |
| R5 | Buang durasi < 60 detik atau > 86.400 detik | Nilai tidak valid |
| R6 | `GROUP BY` stasiun × slot dalam grid lengkap; net flow, kosong/penuh | Struktur tabel fakta |
| R7 | `LEFT JOIN` cuaca per jam (arsip diutamakan atas forecast) | Fitur prediksi |

Lapisan hasil: **silver** = `elt.trips_clean`, `elt.gbfs_status_clean` (data bersih), **gold** = `elt.dim_station`, `elt.weather_hourly`, `elt.fact_station_30min`.

### 4.1 Data inspection pada raw data yang tersimpan (SQL)

In [ ]:
print("Masalah pada raw trip data:")
masalah_trip = INS.masalah_trips(conn, b_trips, BULAN)
display(masalah_trip)
print("Struktur, tipe tersimpan, dan nilai kosong per atribut raw trip data:")
display(INS.struktur_trips(conn, b_trips))
print("Masalah pada raw GBFS station_information:")
display(INS.masalah_gbfs(conn))
print("Ringkasan raw GBFS station_status:")
display(INS.ringkasan_status_gbfs(conn))
display(L.query_df(conn, """select count(*) as jam_cuaca_raw, count(distinct hour_ts) as jam_unik,
        count(*) filter (where temperature_2m is null) as suhu_kosong from raw_elt.weather where batch_id = %s""", (b_w,)))

### 4.2 Cleaning dan transformation (SQL di PostgreSQL)

In [ ]:
# Contoh perintah SQL transformasi yang dijalankan di database (R3, R4, R5 pada trip data):
print(TR.SQL_T0[:900]); print("..."); print(TR.SQL_T2.strip()); print(TR.SQL_T3.strip())

In [ ]:
id_log_awal = int(L.query_df(conn, "select coalesce(max(id), 0) as m from elt.transform_log")["m"].iloc[0])
sebelum_raw = run_elt._hash_raw(conn, b_trips)           # sidik jari raw sebelum Transform

print("Proses Transform dimulai (SQL dijalankan di PostgreSQL)")
ring_trips = TR.transform_trips(conn, BULAN, b_trips)
ring_gbfs = TR.transform_gbfs(conn)
conn.commit()
print("Data berhasil ditransformasi dan disimpan ke schema elt")
display(L.query_df(conn, """select langkah, baris_masuk, baris_keluar, baris_ditolak from elt.transform_log
                            where id > %s order by id""", (id_log_awal,)))
display(L.query_df(conn, "select * from elt.trips_clean order by started_at limit 5"))
display(L.query_df(conn, "select * from elt.fact_station_30min where trips_keluar > 0 order by waktu, station_id limit 5"))

### 4.3 Data validation (SQL pada hasil transformasi)

In [ ]:
def skalar(sql, par=None):
    return L.query_df(conn, sql, par).iloc[0, 0]

p = {"a": awal_bln.to_pydatetime(), "b": akhir_bln.to_pydatetime()}
FT = "elt.fact_station_30min"
n_slot = len(pd.date_range(awal_bln, akhir_bln, freq="30min", inclusive="left"))
n_st = int(skalar(f"select count(distinct station_id) from {FT} where trips_keluar is not null and waktu >= %(a)s and waktu < %(b)s", p))
n_trip_fact = int(skalar(f"select count(*) from {FT} where trips_keluar is not null and waktu >= %(a)s and waktu < %(b)s", p))
validasi = []
def cek(nama, ok, aktual):
    validasi.append({"Pemeriksaan": nama, "Hasil aktual": aktual, "Status": "Berhasil" if ok else "Gagal"})

cek("Jumlah baris = stasiun x slot 30 menit", n_trip_fact == n_st * n_slot, f"{n_trip_fact:,} = {n_st} x {n_slot:,}")
dup = int(skalar(f"select count(*) - count(distinct (station_id, waktu)) from {FT}"))
cek("Tidak ada duplikat (station_id, waktu)", dup == 0, f"{dup} duplikat")
sk, cs = int(skalar(f"select sum(trips_keluar) from {FT} where waktu >= %(a)s and waktu < %(b)s", p)), int(skalar("select count(*) from elt.trips_clean where bulan = %s and start_sid is not null", (BULAN,)))
cek("Total trips_keluar = trip bersih berstasiun asal valid", sk == cs, f"{sk:,} vs {cs:,}")
sm = int(skalar(f"select sum(trips_masuk) from {FT} where waktu >= %(a)s and waktu < %(b)s", p))
cm = int(skalar("select count(*) from elt.trips_clean where bulan = %s and end_sid is not null and ended_at < %s", (BULAN, akhir_bln.to_pydatetime())))
cek("Total trips_masuk = trip bersih bertujuan valid yang berakhir di bulan ini", sm == cm, f"{sm:,} vs {cm:,}")
nf = int(skalar(f"select count(*) from {FT} where trips_keluar is not null and net_flow_trip <> trips_masuk - trips_keluar"))
cek("net_flow_trip = trips_masuk - trips_keluar", nf == 0, f"{nf} baris tidak konsisten")
sl = int(skalar(f"select count(*) from {FT} where extract(minute from waktu) not in (0, 30) or extract(second from waktu) <> 0"))
cek("Semua waktu tepat di awal slot 30 menit", sl == 0, f"{sl} baris menyimpang")
ng = int(skalar(f"select count(*) from {FT} where trips_keluar < 0 or trips_masuk < 0"))
cek("Tidak ada nilai negatif pada jumlah perjalanan", ng == 0, f"{ng} baris negatif")
kos = int(skalar(f"""select count(*) from {FT} where trips_keluar is not null and (suhu_c is null or hujan_mm is null or angin_kmh is null
                      or trips_masuk is null or net_flow_trip is null)"""))
cek("Kolom inti trip tidak memiliki nilai kosong (termasuk cuaca)", kos == 0, f"{kos} baris dengan sel kosong")
tp = L.query_df(conn, "select column_name, data_type from information_schema.columns where table_schema='elt' and table_name='fact_station_30min'").set_index("column_name")["data_type"]
cek("Tipe data: waktu = timestamp, jumlah = integer", tp["waktu"].startswith("timestamp") and tp["trips_keluar"] == "integer", f"waktu: {tp['waktu']}, trips_keluar: {tp['trips_keluar']}")
bk = int(skalar(f"select count(*) from {FT} where station_id !~ '^(JC|HB)'"))
cek("Stasiun hanya Jersey City / Hoboken", bk == 0, f"{int(skalar(f'select count(distinct station_id) from {FT}'))} stasiun")
gs = int(skalar(f"select count(*) from {FT} where n_snapshot is not null and (kosong <> (sepeda_tersedia = 0) or penuh <> (dok_tersedia = 0))"))
cek("GBFS: kosong/penuh konsisten dengan jumlah sepeda/dok", gs == 0, f"{int(skalar(f'select count(*) from {FT} where n_snapshot is not null'))} baris diperiksa")
bd = int(skalar("""select count(*) from elt.fact_station_30min e join public.fact_station_30min p using (station_id, waktu)
                   where e.waktu >= %(a)s and e.waktu < %(b)s and (e.trips_keluar <> p.trips_keluar or e.trips_masuk <> p.trips_masuk
                     or e.net_flow_trip <> p.net_flow_trip or e.suhu_c is distinct from p.suhu_c)""", p))
n_etl = int(skalar("select count(*) from public.fact_station_30min where trips_keluar is not null and waktu >= %(a)s and waktu < %(b)s", p))
cek("Hasil ELT sama dengan hasil pipeline ETL Minggu 10 (bulan yang sama)", bd == 0 and n_etl == n_trip_fact, f"{n_trip_fact:,} baris ELT vs {n_etl:,} baris ETL, {bd} selisih nilai")
hasil_validasi = pd.DataFrame(validasi)
display(hasil_validasi)
assert (hasil_validasi["Status"] == "Berhasil").all(), "Ada pemeriksaan validasi yang gagal"

### 4.4 Perbandingan kondisi data sebelum dan sesudah transformasi

In [ ]:
n_raw = int(skalar("select count(*) from raw_elt.trips where batch_id = %s", (b_trips,)))
n_clean = int(skalar("select count(*) from elt.trips_clean where bulan = %s", (BULAN,)))
mt = masalah_trip.set_index("pemeriksaan")["jumlah"]
dur_buang = n_raw - n_clean
sel_kosong_raw = int(r_trips["sel_kosong"])
bandingkan = pd.DataFrame([
    ["Jumlah baris", f"{n_raw:,} perjalanan (raw)", f"{n_clean:,} perjalanan bersih, lalu {n_trip_fact:,} baris agregat", "Pembersihan lalu agregasi ke stasiun x slot 30 menit"],
    ["Jumlah kolom", 13, int(skalar("select count(*) from information_schema.columns where table_schema='elt' and table_name='fact_station_30min'")), "Atribut perjalanan diganti atribut agregat, kalender, cuaca, dan status GBFS"],
    ["Sel kosong", f"{sel_kosong_raw:,}", f"{kos} (kolom inti)", "Perjalanan tanpa stasiun tidak masuk agregat stasiun"],
    ["Duplikat ride_id", f"{int(mt['ride_id duplikat (baris berlebih)']):,}", "0", "Duplikat dibuang (R4)"],
    ["Durasi > 24 jam / < 1 menit", f"{int(mt['Durasi > 24 jam']):,} / {int(mt['Durasi < 1 menit']):,}", "0", "Dibuang (R5)"],
    ["Waktu tidak valid", f"{int(mt['Waktu tidak dapat dikonversi']):,}", "0", "Dibuang (R3)"],
    ["Tipe started_at / ended_at", "text", "timestamp -> waktu (awal slot)", "Konversi tipe data (R3)"],
], columns=["Aspek", "Sebelum (raw)", "Sesudah (elt)", "Keterangan"])
display(bandingkan)
print(f"Selisih {dur_buang:,} baris = waktu tidak valid {int(mt['Waktu tidak dapat dikonversi'])} + duplikat {int(mt['ride_id duplikat (baris berlebih)'])} "
      f"+ durasi tidak wajar {int(mt['Durasi > 24 jam']) + int(mt['Durasi < 1 menit'])} + di luar bulan {int(mt['Mulai di luar bulan file'])}")
assert dur_buang == int(mt["Waktu tidak dapat dikonversi"]) + int(mt["ride_id duplikat (baris berlebih)"]) + int(mt["Durasi > 24 jam"]) + int(mt["Durasi < 1 menit"]) + int(mt["Mulai di luar bulan file"])

### 4.5 Bukti raw data tetap tersedia dan terpisah dari hasil transformasi

In [ ]:
sesudah_raw = run_elt._hash_raw(conn, b_trips)
print(f"Raw trips sebelum Transform : {sebelum_raw[0]:,} baris, sidik jari {sebelum_raw[1]}")
print(f"Raw trips sesudah Transform : {sesudah_raw[0]:,} baris, sidik jari {sesudah_raw[1]}")
print("Raw tidak berubah / tidak tertimpa:", sebelum_raw == sesudah_raw)
assert sebelum_raw == sesudah_raw
# Baris yang dibuang oleh Transform masih ada di raw (raw tetap utuh) tetapi tidak ada di data bersih:
print("Perjalanan > 24 jam pada raw (masih tersimpan):", int(mt["Durasi > 24 jam"]), "| pada elt.trips_clean: 0")
display(L.query_df(conn, """
    select 'raw_elt' as schema, 'trips' as tabel, count(*) as baris from raw_elt.trips
    union all select 'raw_elt', 'gbfs_station_status', count(*) from raw_elt.gbfs_station_status
    union all select 'raw_elt', 'weather', count(*) from raw_elt.weather
    union all select 'elt', 'trips_clean', count(*) from elt.trips_clean
    union all select 'elt', 'gbfs_status_clean', count(*) from elt.gbfs_status_clean
    union all select 'elt', 'dim_station', count(*) from elt.dim_station
    union all select 'elt', 'weather_hourly', count(*) from elt.weather_hourly
    union all select 'elt', 'fact_station_30min', count(*) from elt.fact_station_30min order by 1, 2"""))

## 5. Integrasi ELT Pipeline

Tiga tahap digabung dalam satu fungsi `run_elt.elt_batch(conn, bulan)` yang berjalan berurutan: **Extract → Load → Transform**.

- Output Extract menjadi input Load; Transform membaca **dari tabel raw yang sudah tersimpan**, bukan dari memori.
- Gerbang urutan: Transform ditolak jika batch raw belum termuat lengkap (`baris_load = baris_extract`).
- Setelah Transform, sidik jari raw dibandingkan dengan sebelumnya; jika berbeda, pipeline gagal.
- Kegagalan pada tahap apa pun membatalkan transaksi (rollback) dan dicatat di `elt.run_log`.

```
Sumber (S3 trip ZIP · GBFS API · Open-Meteo)
  -> EXTRACT  (elt/extract.py)  : ambil apa adanya (teks / JSON)
  -> LOAD     (elt/load.py)     : COPY ke raw_elt.* (append-only) + load_batch
  -> TRANSFORM(elt/transform.py): SQL di PostgreSQL: raw_elt -> elt.trips_clean/gbfs_status_clean (silver) -> elt.fact_station_30min (gold)
```

In [ ]:
hasil_elt = run_elt.elt_batch(conn, BULAN)
print()
print({k: v for k, v in hasil_elt.items() if k not in ("trips", "gbfs")})
display(L.query_df(conn, """select id, job, mulai, selesai, status, baris_extract as extract, baris_load as load,
                            baris_transform as transform, pesan from elt.run_log order by id desc limit 5"""))

## 6. Testing dan validasi

In [ ]:
hasil_pytest = subprocess.run([sys.executable, "-m", "pytest", "-v", "--no-header", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(hasil_pytest.stdout[-3000:])

In [ ]:
uji = []
def uji_tambah(tahap, aspek, harapan, aktual, ok):
    uji.append({"Tahapan": tahap, "Aspek yang diuji": aspek, "Hasil yang diharapkan": harapan, "Hasil aktual": aktual, "Status": "Berhasil" if ok else "Gagal"})

wajib = set(X.KOLOM_TRIPS)
# EXTRACT
uji_tambah("Extract", "Akses sumber data", "Ketiga sumber dapat diakses", "S3, GBFS, Open-Meteo berhasil", len(trips_raw) > 0 and len(status_rec) > 0 and len(cuaca_raw) > 0)
uji_tambah("Extract", "Pembacaan data", "Data berhasil diambil", f"{len(trips_raw):,} trip, {len(info_rec):,} stasiun GBFS, {len(cuaca_raw):,} jam cuaca", len(trips_raw) > 0)
uji_tambah("Extract", "Jumlah dan struktur sesuai sumber", "13 kolom trip; baris sama dengan file sumber", f"{trips_raw.shape[1]} kolom, {len(trips_raw):,} baris", wajib <= set(trips_raw.columns) and trips_raw.shape[1] == 13)
uji_tambah("Extract", "Data tidak diubah", "Semua nilai berupa teks asli (tanpa konversi)", f"tipe: {sorted(set(trips_raw.dtypes.astype(str)))}", set(trips_raw.dtypes.astype(str)) == {"object"})
# LOAD
uji_tambah("Load", "Koneksi ke target storage", "Program terhubung ke Supabase", "terhubung (bagian 1)", conn.closed == 0)
uji_tambah("Load", "Penyimpanan raw data", "Data mentah berhasil disimpan", f"{n_trips:,} baris trips di raw_elt.trips", n_trips == len(trips_raw))
uji_tambah("Load", "Jumlah dan struktur sesuai ekstraksi", "Jumlah baris tersimpan = hasil ekstraksi", f"{n_trips:,} = {len(trips_raw):,}; GBFS {n_s:,} = {len(status_rec):,}", bool(cek_raw["Sama"].all()))
uji_tambah("Load", "Pembacaan kembali", "Raw data dapat diakses dan isinya identik", f"{len(baca_penuh):,} baris terbaca, identik: {identik}", identik)
# TRANSFORM
uji_tambah("Transform", "Proses cleaning", "Duplikat, durasi tidak valid, waktu tidak valid dibersihkan", f"{n_raw:,} -> {n_clean:,} perjalanan bersih ({dur_buang} dibuang)", n_clean == n_raw - dur_buang)
uji_tambah("Transform", "Transformasi sesuai kriteria", "Seluruh pemeriksaan validasi lolos", f"{int((hasil_validasi['Status']=='Berhasil').sum())}/{len(hasil_validasi)} lolos", bool((hasil_validasi["Status"] == "Berhasil").all()))
uji_tambah("Transform", "Penyimpanan hasil", "Data hasil transformasi tersimpan di schema elt", f"{n_trip_fact:,} baris di elt.fact_station_30min", n_trip_fact == n_st * n_slot)
uji_tambah("Transform", "Raw tidak tertimpa", "Raw tetap tersedia dan tidak berubah", f"{sesudah_raw[0]:,} baris, sidik jari sama: {sebelum_raw == sesudah_raw}", sebelum_raw == sesudah_raw and sesudah_raw[0] == len(trips_raw))
# END-TO-END
urut = L.query_df(conn, """select (select max(loaded_at) from raw_elt.load_batch) as raw_terakhir_dimuat,
                                  (select max(run_ts) from elt.transform_log) as transform_terakhir""")
uji_tambah("End-to-End", "Urutan Extract -> Load -> Transform", "Transform berjalan setelah raw tersimpan", f"raw dimuat {urut.iloc[0,0]:%H:%M:%S} <= transform {urut.iloc[0,1]:%H:%M:%S}", urut.iloc[0, 0] <= urut.iloc[0, 1])
st_run = L.query_df(conn, "select status from elt.run_log order by id desc limit 1").iloc[0, 0]
uji_tambah("End-to-End", "Eksekusi pipeline", "Seluruh tahapan berjalan tanpa galat", f"status log: {st_run}; raw_tidak_berubah={hasil_elt['raw_tidak_berubah']}", st_run == "sukses" and hasil_elt["raw_tidak_berubah"])
n_trip_fact2 = int(skalar(f"select count(*) from {FT} where trips_keluar is not null and waktu >= %(a)s and waktu < %(b)s", p))
uji_tambah("End-to-End", "Run ulang (idempotent)", "Jumlah baris raw trips dan fakta tidak bertambah", f"raw {int(skalar('select count(*) from raw_elt.trips')):,}; fakta {n_trip_fact2:,} = {n_trip_fact:,}", n_trip_fact2 == n_trip_fact and int(skalar("select count(*) from raw_elt.trips")) == len(trips_raw))
uji_tambah("End-to-End", "Raw dan hasil terpisah", "Dua schema berbeda berisi data berbeda", "raw_elt.trips (text) vs elt.fact_station_30min (agregat)", int(skalar("select count(*) from information_schema.tables where table_schema='raw_elt'")) >= 5)
tabel_uji = pd.DataFrame(uji)
display(tabel_uji)
print("Ringkasan:", tabel_uji["Status"].value_counts().to_dict())

jumlah_tahap = pd.DataFrame([
    ["Setelah Extract", f"{len(trips_raw):,}", "Data awal dari sumber (perjalanan pada file trip data 202610)"],
    ["Setelah Load", f"{n_raw:,}", "Raw data yang tersimpan di raw_elt.trips (sama dengan hasil ekstraksi)"],
    ["Setelah Transform", f"{n_trip_fact2:,}", f"{n_clean:,} perjalanan bersih diagregasi menjadi {n_st} stasiun x {n_slot:,} slot (elt.fact_station_30min)"],
], columns=["Tahapan", "Jumlah baris", "Keterangan"])
display(jumlah_tahap)

## 7. Dataset

In [ ]:
# Raw data yang dimuat ke target storage (13 kolom sumber, urutan baris asli)
with open("raw_dataset.csv", "w") as f, conn.cursor() as cur:
    cur.copy_expert("""COPY (select ride_id, rideable_type, started_at, ended_at, start_station_name, start_station_id, end_station_name,
        end_station_id, start_lat, start_lng, end_lat, end_lng, member_casual from raw_elt.trips where batch_id = %d order by row_no)
        TO STDOUT WITH CSV HEADER""" % b_trips, f)
# Dataset hasil transformasi yang tersimpan di schema elt
with open("transformed_dataset.csv", "w") as f, conn.cursor() as cur:
    cur.copy_expert("COPY (select * from elt.fact_station_30min order by station_id, waktu) TO STDOUT WITH CSV HEADER", f)
for nama in ("raw_dataset.csv", "transformed_dataset.csv"):
    n_csv = sum(1 for _ in open(nama)) - 1
    print(f"{nama}: {n_csv:,} baris, {os.path.getsize(nama)/1e6:.1f} MB")
display(pd.read_csv("raw_dataset.csv", nrows=3, dtype=str))
display(pd.read_csv("transformed_dataset.csv", nrows=3))

In [ ]:
# Simpan ke Google Drive agar mudah diunduh
from google.colab import drive
drive.mount("/content/drive")
tujuan = "/content/drive/MyDrive/MATPIL BIG DATA/tugas_minggu11"
os.makedirs(tujuan, exist_ok=True)
for nama in ("raw_dataset.csv", "transformed_dataset.csv"):
    subprocess.run(["gzip", "-kf", nama], check=True)
    subprocess.run(["cp", nama + ".gz", tujuan], check=True)
print(os.listdir(tujuan))

In [ ]:
conn.close()
print('Koneksi ditutup. Pipeline selesai.')